# Medallion demo, part 2: Bronze, Silver, Gold in DuckLake

This notebook loads **one load date** from the Bronze Parquet files (`01-medallion-generator.ipynb`) through a strict medallion architecture. Everything
is processed with DuckDB SQL inside **DuckLake** (the Azure DuckLake of `azure.config.json`: data in the Azure storage account, metadata in Azure Database for PostgreSQL); every layer is a DuckLake schema.

| Layer | Schema | What happens |
|---|---|---|
| Bronze | `bronze` | The raw Parquet extract of the load date is copied **unchanged** into a DuckLake table, plus `_load_date` and `_source_file` for lineage. New source columns are added with `ALTER TABLE ... ADD COLUMN` (schema evolution). |
| Silver | `silver` | Cleaned, typed, de-duplicated, conformed. Current state per business key via `MERGE`, deletes (hard and soft), history tables for customers and products, a **quarantine table** `silver.rejects` for invalid rows. |
| Gold | `gold` | Star schema: `fact_sales`, `fact_order`, `fact_payment`, `fact_return` and the dimensions `dim_date`, `dim_customer` and `dim_product` (**SCD type 2**), `dim_store`, `dim_supplier`, `dim_promotion`, `dim_payment_method`, plus five reporting **views** (`v_sales_daily`, `v_sales_by_category_month`, `v_store_performance_month`, `v_customer_value`, `v_payment_method_month`). |

**Incremental and repeatable (idempotent).** Set `LOAD_DATE` in the first cell and run the notebook from top to bottom. A day can be run again
and again with the same result: Bronze replaces the rows of the load date, Silver and Gold recompute only the business keys that the load date touched, directly from
the full Bronze history (so late corrections and re-runs of older days are safe), and every statement is a replace or a `MERGE`, never a blind append. The cells
at the end prove it with row counts and content hashes.

Each step is a function in its own cell, followed by a cell that runs it for `LOAD_DATE`. To load **the whole month day by day in a loop**, set `LOAD_WHOLE_MONTH = True` in the first cell and run the section "Load the whole month in a loop".

**Azure.** This notebook belongs to the Azure DuckLake (the one that aantje creates with *Create Azure DuckLake...*). It reads the connection data from
`azure.config.json` next to it (storage account, container, PostgreSQL server, sign-in mode; no secrets) through `azure_lake.py`. Fill the file in once; all three Azure notebooks use it.
The PostgreSQL password comes from the environment variable `OOG_AZURE_PG_PASSWORD` or a prompt; the storage account is reached with your Microsoft Entra sign-in (a code that you enter in the browser, or a service principal). The public IP address of this machine has to be allowed in the firewall of the PostgreSQL server (aantje: *Allow My IP*).

In [ ]:
# Once per kernel: the Microsoft Entra sign-in. If the import fails afterwards, restart the kernel and run the cells again.
%pip install -q azure-identity

In [ ]:
from datetime import date, timedelta

import azure_lake

LOAD_DATE = date(2026, 9, 1)           # the day to load; use the next day for the next incremental load
BRONZE = azure_lake.bronze_root()      # root of the Bronze Parquet files written by the generator notebook (azure.config.json)
LOAD_WHOLE_MONTH = False               # True: load every day found in the Bronze folder, one after the other (cell "Load the whole month in a loop")
SKIP_LOADED_DAYS = True                # in that loop: skip days that are already loaded (False: load every day again)
BACKFILL_FROM = None                   # alternative to the whole month: a date range for the same loop, e.g. date(2026, 9, 1)
BACKFILL_TO = None                     # ... and date(2026, 9, 30)
print("Load date:", LOAD_DATE, "| whole month loop:", "ON" if LOAD_WHOLE_MONTH else "off")

## Connection and helpers

`azure_lake.connect()` attaches the Azure DuckLake (`lakeName` in `azure.config.json`) as default catalog. The helper functions below inspect tables, create them from a query
(`ensure_table`, which also adds columns that appeared in the source), run several statements as one DuckLake transaction (`tx`, one snapshot), and write
a log row per step into `ops.load_log`.

In [ ]:
import time
from contextlib import contextmanager

import azure_lake

CAT = azure_lake.lake_name()
cat = azure_lake.quote_ident(CAT)
lake = azure_lake.connect()   # attaches the Azure DuckLake and makes it the default catalog


def set_memory_limit(connection, fraction=0.6):
    """DuckDB sizes its memory by the machine, not by the container limit. Without a limit the container's OOM killer ends the kernel on big days;
    with it DuckDB spills to disk instead."""
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            value = open(path).read().strip()
        except OSError:
            continue
        if value.isdigit() and int(value) < 1 << 40:
            limit_mb = int(int(value) / 1024 / 1024 * fraction)
            connection.execute(f"SET memory_limit = '{limit_mb}MB'").fetchall()
            print(f"DuckDB memory limit: {limit_mb:,} MB (container limit {int(value) / 1024 ** 3:.0f} GiB)")
            return


set_memory_limit(lake)


def q(sql):
    """Run a statement and return all rows."""
    return lake.execute(sql).fetchall()


def fq(schema, table):
    return f"{cat}.{azure_lake.quote_ident(schema)}.{azure_lake.quote_ident(table)}"


def qi(name):
    return azure_lake.quote_ident(name)


def has_table(schema, table):
    return q(f"SELECT count(*) FROM information_schema.tables WHERE table_catalog = '{CAT}' AND table_schema = '{schema}' AND table_name = '{table}'")[0][0] > 0


def table_cols(schema, table):
    return [r[0] for r in q(f"SELECT column_name FROM information_schema.columns WHERE table_catalog = '{CAT}' AND table_schema = '{schema}' "
                            f"AND table_name = '{table}' ORDER BY ordinal_position")]


def describe(select_sql):
    return [(r[0], r[1]) for r in q(f"DESCRIBE {select_sql}")]


def ensure_table(schema, table, select_sql):
    """Create the table from the shape of select_sql, or add the columns it does not have yet. Returns the added columns."""
    if not has_table(schema, table):
        q(f"CREATE TABLE {fq(schema, table)} AS SELECT * FROM ({select_sql}) WHERE false")
        return []
    have = set(table_cols(schema, table))
    added = []
    for name, typ in describe(select_sql):
        if name not in have:
            q(f"ALTER TABLE {fq(schema, table)} ADD COLUMN {qi(name)} {typ}")
            added.append(name)
    if added:
        print(f"   schema evolution: {schema}.{table} + {added}")
    return added


@contextmanager
def tx():
    """One DuckLake transaction = one snapshot."""
    q("BEGIN TRANSACTION")
    try:
        yield
    except BaseException:
        q("ROLLBACK")
        raise
    q("COMMIT")


def merge_sql(target, source, keys, cols, matched_cond=None):
    keys = [keys] if isinstance(keys, str) else keys
    on = " AND ".join(f"tgt.{qi(k)} = src.{qi(k)}" for k in keys)
    sets = ", ".join(f"{qi(c)} = src.{qi(c)}" for c in cols if c not in keys)
    cond = f" AND {matched_cond}" if matched_cond else ""
    return (f"MERGE INTO {target} AS tgt USING {source} AS src ON {on} "
            f"WHEN MATCHED{cond} THEN UPDATE SET {sets} "
            f"WHEN NOT MATCHED THEN INSERT ({', '.join(qi(c) for c in cols)}) VALUES ({', '.join('src.' + qi(c) for c in cols)})")


def temp_cols(name):
    return [c for c, _ in describe(f"SELECT * FROM {name}")]


def log_step(layer, table, d, rows_in=0, rows_out=0, rejected=0, deleted=0, t0=None, note=""):
    secs = round(time.time() - t0, 1) if t0 else None
    q(f"DELETE FROM {fq('ops', 'load_log')} WHERE layer = '{layer}' AND table_name = '{table}' AND load_date = DATE '{d}'")
    q(f"INSERT INTO {fq('ops', 'load_log')} VALUES ('{layer}', '{table}', DATE '{d}', {rows_in}, {rows_out}, {rejected}, {deleted}, "
      f"{'NULL' if secs is None else secs}, '{note}', now())")
    print(f"{layer:7} {table:16} in={rows_in:>9,} out={rows_out:>9,} rejected={rejected:>6,} deleted={deleted:>6,} {'' if secs is None else f'{secs:6.1f}s'} {note}")

## Layer schemas and control tables

`bronze`, `silver` and `gold` are the three layers. `ops` holds the load log (one row per layer, table and load date; a re-run replaces the row). `silver.rejects` is
the quarantine for rows that fail the Silver rules.

In [ ]:
for schema in ("bronze", "silver", "gold", "ops"):
    q(f"CREATE SCHEMA IF NOT EXISTS {cat}.{schema}")

q(f"""CREATE TABLE IF NOT EXISTS {fq('ops', 'load_log')} (
    layer VARCHAR, table_name VARCHAR, load_date DATE, rows_in BIGINT, rows_out BIGINT, rows_rejected BIGINT, rows_deleted BIGINT,
    seconds DOUBLE, note VARCHAR, finished_at TIMESTAMP)""")

q(f"""CREATE TABLE IF NOT EXISTS {fq('silver', 'rejects')} (
    source_table VARCHAR, record_key VARCHAR, reason VARCHAR, updated_at TIMESTAMP, load_date DATE, payload VARCHAR)""")

print([r[0] for r in q(f"SELECT schema_name FROM information_schema.schemata WHERE catalog_name = '{CAT}' ORDER BY 1")])

## Cleaning macros

The reusable cleaning rules are **macros stored in DuckLake**, in the schema `silver` (`CREATE OR REPLACE MACRO silver.proper(x) AS ...`). DuckLake keeps macros in the catalog like tables: they outlive the session,
every client of the lake can call them (for example aantje or a report) and each change is a snapshot. The cleaning SQL below calls them with the schema, for example `silver.iso2(country)`.
The rules: proper case, country names to ISO codes, phone numbers to international format, date parsing for three source formats and an e-mail check.

In [ ]:
SILVER_MACROS = {
    "proper": ("x", "upper(left(trim(x), 1)) || lower(substr(trim(x), 2))"),
    "iso2": ("c", """(CASE lower(trim(c))
        WHEN 'de' THEN 'DE' WHEN 'deutschland' THEN 'DE' WHEN 'germany' THEN 'DE'
        WHEN 'at' THEN 'AT' WHEN 'österreich' THEN 'AT' WHEN 'austria' THEN 'AT'
        WHEN 'ch' THEN 'CH' WHEN 'switzerland' THEN 'CH' WHEN 'schweiz' THEN 'CH'
        WHEN 'nl' THEN 'NL' WHEN 'fr' THEN 'FR' WHEN 'pl' THEN 'PL' WHEN 'cz' THEN 'CZ' WHEN 'it' THEN 'IT'
        ELSE NULL END)"""),
    "norm_phone": ("p", """(CASE
        WHEN regexp_replace(p, '[^0-9]', '', 'g') LIKE '49%' THEN '+' || regexp_replace(p, '[^0-9]', '', 'g')
        WHEN regexp_replace(p, '[^0-9]', '', 'g') LIKE '0%' THEN '+49' || substr(regexp_replace(p, '[^0-9]', '', 'g'), 2)
        ELSE NULL END)"""),
    "parse_date": ("s", "CAST(COALESCE(try_strptime(s, '%Y-%m-%d'), try_strptime(s, '%d.%m.%Y'), try_strptime(s, '%m/%d/%Y')) AS DATE)"),
    "valid_email": ("e", "(CASE WHEN lower(trim(e)) LIKE '_%@_%._%' THEN lower(trim(e)) END)"),
}

for name, (parameter, body) in SILVER_MACROS.items():
    q(f"CREATE OR REPLACE MACRO {fq('silver', name)}({parameter}) AS {body}")

print([r[0] for r in q(f"SELECT function_name FROM duckdb_functions() WHERE database_name = '{CAT}' AND schema_name = 'silver' ORDER BY 1")], "stored in", f"{CAT}.silver")
print(q("SELECT silver.proper('  mUeLLer '), silver.iso2('Deutschland'), silver.norm_phone('(0170) 1234567'), silver.parse_date('16.09.1979'), silver.valid_email(' A@B.de ')"))

## Bronze: raw Parquet to DuckLake

`bronze_load` reads the extract of the load date from Azure Storage and copies it **unchanged** into `bronze.<table>`, adding `_load_date` and `_source_file`.
If the extract has a column that the table does not have yet (the `loyalty_tier` of the customers from 15 September on), `ensure_table` adds it first.
The rows of the load date are replaced in one transaction (`DELETE` + `INSERT`), so running a day twice never duplicates anything. Days without a file (for example the weekly snapshots) are skipped.

In [ ]:
BRONZE_TABLES = ["customers", "addresses", "products", "suppliers", "stores", "promotions", "orders", "order_items", "payments", "returns"]


def bronze_load(table, d):
    t0 = time.time()
    path = f"{BRONZE}/{table}/load_date={d}/{table}.parquet"
    # a wildcard makes glob() list the folder; glob() of an exact storage path always "finds" it
    if q(f"SELECT count(*) FROM glob('{BRONZE}/{table}/load_date={d}/{table}*.parquet')")[0][0] == 0:
        print(f"bronze  {table:16} no file for {d}, skipped")
        return 0
    source = (f"SELECT * EXCLUDE (filename), DATE '{d}' AS _load_date, filename AS _source_file "
              f"FROM read_parquet('{path}', filename = true, hive_partitioning = false)")
    ensure_table("bronze", table, source)
    expected = q(f"SELECT count(*) FROM read_parquet('{path}', hive_partitioning = false)")[0][0]
    target = fq("bronze", table)
    with tx():
        q(f"DELETE FROM {target} WHERE _load_date = DATE '{d}'")
        q(f"INSERT INTO {target} BY NAME {source}")
    loaded = q(f"SELECT count(*) FROM {target} WHERE _load_date = DATE '{d}'")[0][0]
    assert loaded == expected, f"{table}: {loaded} rows loaded, {expected} in the file"
    log_step("bronze", table, d, rows_in=expected, rows_out=loaded, t0=t0)
    return loaded

In [ ]:
bronze_load("customers", LOAD_DATE)

In [ ]:
bronze_load("addresses", LOAD_DATE)

In [ ]:
bronze_load("products", LOAD_DATE)

In [ ]:
bronze_load("suppliers", LOAD_DATE)

In [ ]:
bronze_load("stores", LOAD_DATE)

In [ ]:
bronze_load("promotions", LOAD_DATE)

In [ ]:
bronze_load("orders", LOAD_DATE)

In [ ]:
bronze_load("order_items", LOAD_DATE)

In [ ]:
bronze_load("payments", LOAD_DATE)

In [ ]:
bronze_load("returns", LOAD_DATE)

## Silver: clean, conform, merge

All Silver tables follow the same procedure (`silver_cdc`), only the cleaning rules differ:

1. **Affected keys:** the business keys that appear in the Bronze rows of the load date.
2. **Versions:** all Bronze rows of these keys that are loaded so far, from **every** load date (de-duplicated per key and `updated_at`). Reading the whole history makes re-runs and
   late corrections safe: the result never depends on the order in which days were loaded, and running an older day again does not undo the updates and deletes that later days brought.
3. **Clean and validate:** the table-specific SQL produces the cleaned columns and a reject reason (or NULL).
4. **Apply to the current-state table:** `MERGE` of the latest valid version per key (only if it is not older than the stored row), plus deletes: physical `DELETE`
   (order items), soft delete with `is_deleted` (orders) or none.
5. **Quarantine:** invalid rows go to `silver.rejects` with the reason and the raw record as JSON (replaced per key, so a corrected record disappears from the quarantine).
6. **History tables** (customers, products) keep every valid version for the SCD type 2 dimensions in Gold.

In [ ]:
SILVER = {}   # name -> spec, filled by the cells below


def silver_cdc(name, d):
    s = SILVER[name]
    t0 = time.time()
    key, bronze, target, hist = s["key"], fq("bronze", s["bronze"]), fq("silver", s["silver"]), s.get("hist")
    if not has_table("bronze", s["bronze"]):
        print(f"silver  {s['silver']:16} bronze table missing, skipped")
        return
    q(f"CREATE OR REPLACE TEMP TABLE _affected AS SELECT DISTINCT {key} AS k FROM {bronze} WHERE _load_date = DATE '{d}'")
    if q("SELECT count(*) FROM _affected")[0][0] == 0:
        print(f"silver  {s['silver']:16} nothing delivered for {d}, skipped")
        return
    q(f"""CREATE OR REPLACE TEMP TABLE _versions AS
          SELECT * EXCLUDE (_rn) FROM (
              SELECT b.*, row_number() OVER (PARTITION BY b.{key}, b.updated_at ORDER BY b._load_date DESC) AS _rn
              FROM {bronze} b WHERE b.{key} IN (SELECT k FROM _affected)) WHERE _rn = 1""")
    cols, joins, reject = s["clean"]()
    q(f"CREATE OR REPLACE TEMP TABLE _clean AS SELECT {cols}, ({reject}) AS _reject, {'l.op' if s.get('has_op', True) else chr(39) + 'I' + chr(39)} AS _op FROM _versions l {joins}")
    q(f"""CREATE OR REPLACE TEMP TABLE _last AS
          SELECT {key} AS k, _op, updated_at AS last_ts FROM _clean
          QUALIFY row_number() OVER (PARTITION BY {key} ORDER BY updated_at DESC, CASE _op WHEN 'D' THEN 1 ELSE 0 END DESC) = 1""")
    q(f"""CREATE OR REPLACE TEMP TABLE _cur AS
          SELECT * EXCLUDE (_reject, _op) FROM _clean
          WHERE _reject IS NULL AND _op <> 'D' AND {key} NOT IN (SELECT k FROM _last WHERE _op = 'D')
          QUALIFY row_number() OVER (PARTITION BY {key} ORDER BY updated_at DESC) = 1""")
    ensure_table("silver", s["silver"], "SELECT * FROM _cur")
    if hist:
        q("CREATE OR REPLACE TEMP TABLE _hist AS SELECT * EXCLUDE (_reject, _op) FROM _clean WHERE _reject IS NULL AND _op <> 'D'")
        ensure_table("silver", hist, "SELECT * FROM _hist")
    cols_cur = temp_cols("_cur")
    deleted = q("SELECT count(*) FROM _last WHERE _op = 'D'")[0][0]
    with tx():
        q(merge_sql(target, "_cur", key, cols_cur, matched_cond="src.updated_at >= tgt.updated_at"))
        if s.get("delete") == "hard":
            q(f"DELETE FROM {target} WHERE {key} IN (SELECT k FROM _last WHERE _op = 'D')")
        elif s.get("delete") == "soft":
            q(f"UPDATE {target} SET is_deleted = true, deleted_at = l.last_ts FROM _last l WHERE {target}.{key} = l.k AND l._op = 'D'")
        if hist:
            q(f"INSERT INTO {fq('silver', hist)} BY NAME SELECT * FROM _hist h "
              f"WHERE NOT EXISTS (SELECT 1 FROM {fq('silver', hist)} x WHERE x.{key} = h.{key} AND x.updated_at = h.updated_at)")
        q(f"DELETE FROM {fq('silver', 'rejects')} WHERE source_table = '{s['bronze']}' AND record_key IN (SELECT CAST(k AS VARCHAR) FROM _affected)")
        q(f"""INSERT INTO {fq('silver', 'rejects')}
              SELECT '{s['bronze']}', CAST(c.{key} AS VARCHAR), c._reject, c.updated_at, l._load_date, CAST(to_json(l) AS VARCHAR)
              FROM _clean c JOIN _versions l ON l.{key} = c.{key} AND l.updated_at = c.updated_at WHERE c._reject IS NOT NULL""")
        if s.get("post"):
            s["post"](d)
    rows_in = q("SELECT count(*) FROM _versions")[0][0]
    rows_out = q("SELECT count(*) FROM _cur")[0][0]
    rejected = q("SELECT count(*) FROM _clean WHERE _reject IS NOT NULL")[0][0]
    log_step("silver", s["silver"], d, rows_in=rows_in, rows_out=rows_out, rejected=rejected, deleted=deleted, t0=t0)

### Silver `customer` and `customer_change`

Cleaning: names in proper case, e-mail lower case and checked, phone numbers in international format, birth dates parsed from three formats (invalid dates and
implausible years become NULL), gender and country mapped to codes, segment lower case. A customer without a valid e-mail **and** without a phone number is quarantined
("no contact data"). A column that only exists in newer extracts (`loyalty_tier`) is picked up automatically.

In [ ]:
def clean_customers():
    tier = ", l.loyalty_tier" if "loyalty_tier" in table_cols("bronze", "customers") else ""
    cols = """
        l.customer_id,
        silver.proper(l.first_name) AS first_name,
        silver.proper(l.last_name) AS last_name,
        silver.valid_email(l.email) AS email,
        silver.norm_phone(l.phone) AS phone,
        CASE WHEN silver.parse_date(l.birth_date) BETWEEN DATE '1900-01-01' AND DATE '2010-12-31' THEN silver.parse_date(l.birth_date) END AS birth_date,
        CASE upper(trim(l.gender)) WHEN 'M' THEN 'M' WHEN 'MALE' THEN 'M' WHEN 'F' THEN 'F' WHEN 'FEMALE' THEN 'F' WHEN 'D' THEN 'D' END AS gender,
        silver.iso2(l.country) AS country,
        lower(trim(l.segment)) AS segment,
        l.registered_at,
        l.updated_at,
        l._load_date AS bronze_load_date""" + tier
    reject = "CASE WHEN silver.valid_email(l.email) IS NULL AND silver.norm_phone(l.phone) IS NULL THEN 'no contact data (no valid e-mail and no phone)' END"
    return cols, "", reject


SILVER["customers"] = dict(key="customer_id", bronze="customers", silver="customer", hist="customer_change", clean=clean_customers)

In [ ]:
silver_cdc("customers", LOAD_DATE)

### Silver `address`

Postal codes are reduced to digits and padded to five digits, city names are in proper case, countries are ISO codes, the address type is lower case.
An address without a postal code is quarantined.

In [ ]:
def clean_addresses():
    cols = """
        l.address_id,
        l.customer_id,
        lower(trim(l.address_type)) AS address_type,
        trim(l.street) AS street,
        trim(l.house_no) AS house_no,
        lpad(regexp_replace(l.postal_code, '[^0-9]', '', 'g'), 5, '0') AS postal_code,
        silver.proper(l.city) AS city,
        silver.iso2(l.country) AS country,
        l.is_primary,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = "CASE WHEN nullif(regexp_replace(l.postal_code, '[^0-9]', '', 'g'), '') IS NULL THEN 'missing postal code' END"
    return cols, "", reject


SILVER["addresses"] = dict(key="address_id", bronze="addresses", silver="address", clean=clean_addresses)

In [ ]:
silver_cdc("addresses", LOAD_DATE)

### Silver `product` and `product_change`

Categories are mapped from their many spellings to the master list, brand names are in proper case, an EAN must have 13 digits, the SKU is upper case.
A product with a price of zero or less is quarantined (data entry bug in the source).

In [ ]:
def clean_products():
    cols = """
        l.product_id,
        upper(trim(l.sku)) AS sku,
        trim(l.product_name) AS product_name,
        CASE lower(trim(l.category)) WHEN 'electronics' THEN 'Electronics' WHEN 'home & garden' THEN 'Home & Garden' WHEN 'fashion' THEN 'Fashion'
             WHEN 'sports' THEN 'Sports' WHEN 'toys' THEN 'Toys' WHEN 'books' THEN 'Books' WHEN 'beauty' THEN 'Beauty' WHEN 'grocery' THEN 'Grocery'
             ELSE 'Other' END AS category,
        silver.proper(l.subcategory) AS subcategory,
        silver.proper(l.brand) AS brand,
        l.supplier_id,
        l.list_price,
        l.cost_price,
        CASE WHEN regexp_matches(l.ean, '^[0-9]{13}$') THEN l.ean END AS ean,
        l.is_active,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = "CASE WHEN l.list_price IS NULL OR l.list_price <= 0 THEN 'non-positive list price' END"
    return cols, "", reject


SILVER["products"] = dict(key="product_id", bronze="products", silver="product", hist="product_change", clean=clean_products)

In [ ]:
silver_cdc("products", LOAD_DATE)

### Silver `supplier`, `store`, `promotion` (weekly snapshots)

These sources deliver full snapshots. The rows are merged like the others; in addition, everything that is **missing from a newer snapshot is marked inactive** (`is_active = false`,
`deactivated_on` = the first snapshot without it) - a soft delete that is detected, not delivered. Rules: supplier e-mail checked, country to ISO code; sqm of a store must be positive;
a promotion discount above 100 % or a validity window that ends before it starts is quarantined.

In [ ]:
def clean_suppliers():
    cols = """
        l.supplier_id,
        trim(l.supplier_name) AS supplier_name,
        silver.iso2(l.country) AS country,
        silver.valid_email(l.contact_email) AS contact_email,
        l.payment_terms_days,
        CASE WHEN l.rating BETWEEN 1 AND 5 THEN l.rating END AS rating,
        upper(trim(l.active_flag)) = 'Y' AS is_active,
        CAST(NULL AS DATE) AS deactivated_on,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = "CASE WHEN nullif(trim(l.supplier_name), '') IS NULL THEN 'missing supplier name' END"
    return cols, "", reject


def deactivate_missing(silver_table, bronze_table, key):
    """Mark every row that is missing from a newer snapshot as inactive, with the date of the first snapshot that no longer has it.
    The result depends only on the snapshots in Bronze, not on the load date: running an older day again changes nothing."""
    def post(d):
        bronze = fq('bronze', bronze_table)
        q(f"""UPDATE {fq('silver', silver_table)} AS t SET is_active = false, deactivated_on = m.gone
              FROM (SELECT k, (SELECT min(_load_date) FROM {bronze} WHERE _load_date > last_seen) AS gone
                    FROM (SELECT {key} AS k, max(_load_date) AS last_seen FROM {bronze} GROUP BY {key})) m
              WHERE t.{key} = m.k AND t.is_active AND m.gone IS NOT NULL""")
    return post


SILVER["suppliers"] = dict(key="supplier_id", bronze="suppliers", silver="supplier", has_op=False, clean=clean_suppliers,
                           post=deactivate_missing("supplier", "suppliers", "supplier_id"))

In [ ]:
silver_cdc("suppliers", LOAD_DATE)

In [ ]:
def clean_stores():
    cols = """
        l.store_id,
        trim(l.store_name) AS store_name,
        lower(trim(l.store_type)) AS store_type,
        silver.proper(l.city) AS city,
        silver.iso2(l.country) AS country,
        lower(trim(l.region)) AS region,
        l.opened_on,
        trim(l.manager_name) AS manager_name,
        CASE WHEN l.sqm > 0 THEN l.sqm END AS sqm,
        l.is_open AS is_open,
        true AS is_active,
        CAST(NULL AS DATE) AS deactivated_on,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = "CASE WHEN nullif(trim(l.store_name), '') IS NULL THEN 'missing store name' END"
    return cols, "", reject


SILVER["stores"] = dict(key="store_id", bronze="stores", silver="store", has_op=False, clean=clean_stores,
                        post=deactivate_missing("store", "stores", "store_id"))

In [ ]:
silver_cdc("stores", LOAD_DATE)

In [ ]:
def clean_promotions():
    cols = """
        l.promotion_id,
        upper(trim(l.promo_code)) AS promo_code,
        trim(l.description) AS description,
        l.discount_pct,
        lower(trim(l.discount_type)) AS discount_type,
        l.valid_from,
        l.valid_to,
        lower(trim(l.channel)) AS channel,
        true AS is_active,
        CAST(NULL AS DATE) AS deactivated_on,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = """CASE WHEN l.discount_pct IS NULL OR l.discount_pct <= 0 OR l.discount_pct > 100 THEN 'discount outside 1-100 percent'
                     WHEN l.valid_to < l.valid_from THEN 'validity window ends before it starts' END"""
    return cols, "", reject


SILVER["promotions"] = dict(key="promotion_id", bronze="promotions", silver="promotion", has_op=False, clean=clean_promotions,
                            post=deactivate_missing("promotion", "promotions", "promotion_id"))

In [ ]:
silver_cdc("promotions", LOAD_DATE)

### Silver `sales_order`

Status values are normalized (and must be one of the five known states, otherwise the row is quarantined), channel and currency are cleaned, a missing customer becomes the
unknown customer `-1`. Orders that the source deletes (`op = 'D'`) are **soft deleted** (`is_deleted`, `deleted_at`) so that Gold can still explain what happened.

In [ ]:
def clean_orders():
    cols = """
        l.order_id,
        COALESCE(l.customer_id, -1) AS customer_id,
        l.store_id,
        l.order_ts,
        lower(trim(l.status)) AS status,
        COALESCE(lower(trim(l.channel)), 'unknown') AS channel,
        upper(trim(l.currency)) AS currency,
        l.order_total_reported,
        l.shipping_cost,
        l.promotion_id,
        false AS is_deleted,
        CAST(NULL AS TIMESTAMP) AS deleted_at,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = """CASE WHEN lower(trim(l.status)) NOT IN ('created', 'paid', 'shipped', 'delivered', 'cancelled') THEN 'unknown order status'
                     WHEN l.order_total_reported IS NULL OR l.order_total_reported < 0 THEN 'invalid order total' END"""
    return cols, "", reject


SILVER["orders"] = dict(key="order_id", bronze="orders", silver="sales_order", clean=clean_orders, delete="soft")

In [ ]:
silver_cdc("orders", LOAD_DATE)

### Silver `order_item`

The largest table. Computed columns: `gross_amount`, `discount_amount`, `net_amount`. Quarantine reasons: quantity zero or negative, price missing or not positive,
and **orphans** (the order does not exist in `silver.sales_order`). Items that the source deletes are **physically deleted**.

In [ ]:
def clean_items():
    cols = """
        l.item_id,
        l.order_id,
        l.product_id,
        l.quantity,
        l.unit_price,
        least(greatest(COALESCE(l.discount_pct, 0), 0), 100) AS discount_pct,
        l.tax_rate,
        CAST(l.quantity * l.unit_price AS DECIMAL(14, 2)) AS gross_amount,
        CAST(round(l.quantity * l.unit_price * least(greatest(COALESCE(l.discount_pct, 0), 0), 100) / 100.0, 2) AS DECIMAL(14, 2)) AS discount_amount,
        CAST(l.quantity * l.unit_price - round(l.quantity * l.unit_price * least(greatest(COALESCE(l.discount_pct, 0), 0), 100) / 100.0, 2) AS DECIMAL(14, 2)) AS net_amount,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    joins = f"LEFT JOIN {fq('silver', 'sales_order')} so ON so.order_id = l.order_id"
    reject = """CASE WHEN l.quantity IS NULL OR l.quantity <= 0 THEN 'non-positive quantity'
                     WHEN l.unit_price IS NULL OR l.unit_price <= 0 THEN 'missing or non-positive unit price'
                     WHEN so.order_id IS NULL THEN 'orphan: order not found' END"""
    return cols, joins, reject


SILVER["order_items"] = dict(key="item_id", bronze="order_items", silver="order_item", clean=clean_items, delete="hard")

In [ ]:
silver_cdc("order_items", LOAD_DATE)

### Silver `payment` and `product_return`

Payments: eleven spellings of the payment method are mapped to six values, the status is lower case, an amount of zero or less is quarantined unless the payment failed.
Returns: a missing reason becomes `unknown`, quantity and refund amount must be positive.

In [ ]:
def clean_payments():
    cols = """
        l.payment_id,
        l.order_id,
        CASE WHEN lower(l.payment_method) LIKE '%visa%' OR lower(l.payment_method) LIKE '%master%' OR lower(l.payment_method) LIKE '%credit%' THEN 'credit_card'
             WHEN lower(l.payment_method) LIKE '%paypal%' THEN 'paypal'
             WHEN lower(l.payment_method) LIKE 'sepa%' THEN 'sepa_direct_debit'
             WHEN lower(l.payment_method) LIKE '%klarna%' THEN 'klarna'
             WHEN lower(l.payment_method) LIKE '%invoice%' THEN 'invoice'
             WHEN lower(l.payment_method) LIKE '%apple%' THEN 'apple_pay'
             ELSE 'other' END AS payment_method,
        l.amount,
        upper(trim(l.currency)) AS currency,
        lower(trim(l.status)) AS status,
        l.paid_at,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = "CASE WHEN (l.amount IS NULL OR l.amount <= 0) AND lower(trim(l.status)) <> 'failed' THEN 'non-positive amount' END"
    return cols, "", reject


SILVER["payments"] = dict(key="payment_id", bronze="payments", silver="payment", clean=clean_payments)

In [ ]:
silver_cdc("payments", LOAD_DATE)

In [ ]:
def clean_returns():
    cols = """
        l.return_id,
        l.item_id,
        l.order_id,
        l.quantity,
        COALESCE(lower(trim(l.reason)), 'unknown') AS reason,
        l.refund_amount,
        lower(trim(l.status)) AS status,
        l.return_ts,
        l.updated_at,
        l._load_date AS bronze_load_date"""
    reject = """CASE WHEN l.quantity IS NULL OR l.quantity <= 0 THEN 'non-positive quantity'
                     WHEN l.refund_amount IS NULL OR l.refund_amount <= 0 THEN 'non-positive refund amount' END"""
    return cols, "", reject


SILVER["returns"] = dict(key="return_id", bronze="returns", silver="product_return", clean=clean_returns)

In [ ]:
silver_cdc("returns", LOAD_DATE)

## Gold: the star schema

**Dimensions**
* `dim_date`: one row per day 2020 to 2030, created once.
* `dim_customer` and `dim_product`: **SCD type 2**. For the customers that the load date touched, all versions are rebuilt from the Silver history tables. A new version only exists when a tracked attribute really changed
  (consecutive identical versions are collapsed); `valid_from`, `valid_to` and `is_current` describe the validity, the first version is valid from 1900-01-01. The surrogate key is a hash of the business key and the version start, so it is stable across re-runs.
* `dim_store`, `dim_supplier`, `dim_promotion`: type 1 (overwrite). `dim_payment_method`: a small static dimension.
* Every dimension has an **unknown member** with key `-1` for facts whose reference is missing (guest orders, orphan products, rejected customers).

**Facts** (always replaced per business key, never appended)
* `fact_sales`: one row per order item with the dimension keys **as of the order time** (`ASOF JOIN` on the SCD2 dimensions).
* `fact_order`, `fact_payment`, `fact_return`: order header, payment and return grain.

In [ ]:
def gold_dim1(name, silver_select, key_sql, unknown_sql, d):
    """Type 1 dimension: MERGE of the changed rows (compared by a row hash), plus the unknown member."""
    t0 = time.time()
    q(f"""CREATE OR REPLACE TEMP TABLE _dim AS
          WITH s AS ({silver_select}), u AS ({unknown_sql})
          SELECT s.*, hash(s) AS row_hash FROM s UNION ALL BY NAME SELECT u.*, hash(u) AS row_hash FROM u""")
    ensure_table("gold", name, "SELECT * FROM _dim")
    cols = temp_cols("_dim")
    target = fq("gold", name)
    with tx():
        q(merge_sql(target, "_dim", key_sql, cols, matched_cond="tgt.row_hash <> src.row_hash"))
    n = q(f"SELECT count(*) FROM {target}")[0][0]
    log_step("gold", name, d, rows_in=q("SELECT count(*) FROM _dim")[0][0], rows_out=n, t0=t0)


def gold_scd2(dim, dim_key, hist, key, attrs, derived, unknown_sql, bronze_table, d):
    """SCD type 2: rebuild all versions of the business keys that the load date touched."""
    t0 = time.time()
    available = set(table_cols("silver", hist))
    attrs = [a for a in attrs if a in available]
    attr_csv = ", ".join(attrs)
    q(f"CREATE OR REPLACE TEMP TABLE _sk AS SELECT DISTINCT {key} AS k FROM {fq('bronze', bronze_table)} WHERE _load_date = DATE '{d}'")
    q(f"""CREATE OR REPLACE TEMP TABLE _dim_new AS
          WITH v AS (SELECT {key}, updated_at, {attr_csv}, hash({attr_csv}) AS _h FROM {fq('silver', hist)} WHERE {key} IN (SELECT k FROM _sk)),
               c AS (SELECT *, lag(_h) OVER (PARTITION BY {key} ORDER BY updated_at) AS _ph,
                             row_number() OVER (PARTITION BY {key} ORDER BY updated_at) AS _rn FROM v),
               k AS (SELECT * FROM c WHERE _ph IS NULL OR _ph <> _h),
               s AS (SELECT *, lead(updated_at) OVER (PARTITION BY {key} ORDER BY updated_at) AS _next FROM k)
          SELECT CAST(hash({key}, epoch_ms(updated_at)) % 4611686018427387904 AS BIGINT) AS {dim_key}, {key}, {attr_csv}, {derived},
                 updated_at AS effective_from,
                 CASE WHEN _rn = 1 THEN TIMESTAMP '1900-01-01' ELSE updated_at END AS valid_from,
                 COALESCE(_next, TIMESTAMP '9999-12-31') AS valid_to,
                 _next IS NULL AS is_current
          FROM s""")
    ensure_table("gold", dim, "SELECT * FROM _dim_new")
    target = fq("gold", dim)
    with tx():
        q(f"DELETE FROM {target} WHERE {key} IN (SELECT k FROM _sk)")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM _dim_new")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM ({unknown_sql}) WHERE NOT EXISTS (SELECT 1 FROM {target} x WHERE x.{dim_key} = -1)")
    log_step("gold", dim, d, rows_in=q("SELECT count(*) FROM _sk")[0][0], rows_out=q("SELECT count(*) FROM _dim_new")[0][0], t0=t0)


def late_dimension_keys(d):
    """Customers and products delivered on the load date. Facts that were loaded earlier and point to the unknown member (-1) for one of
    them are recomputed too: its dimension row may exist only now (a late arriving dimension). Without this a fact would depend on the
    order in which the days were loaded."""
    return (f"SELECT customer_id FROM {fq('bronze', 'customers')} WHERE _load_date = DATE '{d}'",
            f"SELECT product_id FROM {fq('bronze', 'products')} WHERE _load_date = DATE '{d}'")

### `dim_date`
Created once (the cell does nothing when the table exists).

In [ ]:
def gold_dim_date():
    if has_table("gold", "dim_date"):
        print("gold    dim_date         exists")
        return
    q(f"""CREATE TABLE {fq('gold', 'dim_date')} AS
          SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key, CAST(d AS DATE) AS date, year(d) AS year, quarter(d) AS quarter, month(d) AS month,
                 strftime(d, '%B') AS month_name, day(d) AS day, isodow(d) AS iso_day_of_week, strftime(d, '%A') AS day_name, weekofyear(d) AS iso_week,
                 isodow(d) >= 6 AS is_weekend, CAST(d AS DATE) = last_day(CAST(d AS DATE)) AS is_month_end
          FROM generate_series(DATE '2020-01-01', DATE '2030-12-31', INTERVAL 1 DAY) t(d)""")
    print("gold    dim_date         created,", q(f"SELECT count(*) FROM {fq('gold', 'dim_date')}")[0][0], "days")


gold_dim_date()

### `dim_payment_method`, `dim_store`, `dim_supplier`, `dim_promotion`

In [ ]:
def gold_dim_payment_method(d):
    methods = ["credit_card", "paypal", "sepa_direct_debit", "klarna", "invoice", "apple_pay", "other"]
    rows = " UNION ALL ".join(f"SELECT {i + 1} AS payment_method_key, '{m}' AS payment_method, '{m.replace('_', ' ').title()}' AS payment_method_name" for i, m in enumerate(methods))
    gold_dim1("dim_payment_method", rows, "payment_method_key",
              "SELECT -1 AS payment_method_key, 'unknown' AS payment_method, 'Unknown' AS payment_method_name", d)


def gold_dim_store(d):
    gold_dim1("dim_store",
              f"SELECT store_id AS store_key, store_id, store_name, store_type, city, country, region, opened_on, manager_name, sqm, is_open, is_active FROM {fq('silver', 'store')}",
              "store_key", "SELECT -1 AS store_key, -1 AS store_id, 'Unknown store' AS store_name, 'unknown' AS store_type", d)


def gold_dim_supplier(d):
    gold_dim1("dim_supplier",
              f"SELECT supplier_id AS supplier_key, supplier_id, supplier_name, country, payment_terms_days, rating, is_active FROM {fq('silver', 'supplier')}",
              "supplier_key", "SELECT -1 AS supplier_key, -1 AS supplier_id, 'Unknown supplier' AS supplier_name", d)


def gold_dim_promotion(d):
    gold_dim1("dim_promotion",
              f"SELECT promotion_id AS promotion_key, promotion_id, promo_code, description, discount_pct, valid_from, valid_to, channel, is_active FROM {fq('silver', 'promotion')}",
              "promotion_key", "SELECT -1 AS promotion_key, -1 AS promotion_id, 'NONE' AS promo_code, 'No promotion' AS description", d)

In [ ]:
gold_dim_payment_method(LOAD_DATE)
gold_dim_store(LOAD_DATE)
gold_dim_supplier(LOAD_DATE)
gold_dim_promotion(LOAD_DATE)

### `dim_customer` and `dim_product` (SCD type 2)

In [ ]:
def gold_dim_customer(d):
    gold_scd2(
        "dim_customer", "customer_key", "customer_change", "customer_id",
        ["first_name", "last_name", "email", "phone", "birth_date", "gender", "country", "segment", "loyalty_tier"],
        "first_name || ' ' || last_name AS full_name, year(birth_date) AS birth_year",
        "SELECT -1 AS customer_key, -1 AS customer_id, 'Unknown' AS first_name, 'customer' AS last_name, 'Unknown customer' AS full_name, "
        "TIMESTAMP '1900-01-01' AS effective_from, TIMESTAMP '1900-01-01' AS valid_from, TIMESTAMP '9999-12-31' AS valid_to, true AS is_current",
        "customers", d)


def gold_dim_product(d):
    gold_scd2(
        "dim_product", "product_key", "product_change", "product_id",
        ["sku", "product_name", "category", "subcategory", "brand", "supplier_id", "list_price", "cost_price", "ean", "is_active"],
        "CASE WHEN list_price > 0 THEN round((list_price - cost_price) / list_price * 100, 1) END AS margin_pct",
        "SELECT -1 AS product_key, -1 AS product_id, 'UNKNOWN' AS sku, 'Unknown product' AS product_name, 'Other' AS category, "
        "TIMESTAMP '1900-01-01' AS effective_from, TIMESTAMP '1900-01-01' AS valid_from, TIMESTAMP '9999-12-31' AS valid_to, true AS is_current",
        "products", d)

In [ ]:
gold_dim_customer(LOAD_DATE)
gold_dim_product(LOAD_DATE)

### Facts

For every fact table the **affected keys** are determined first (the rows delivered on the load date, plus everything that depends on them, for example all items of an order whose status changed).
Rows of earlier days that still point to the unknown member (-1) for a customer or product delivered today are included, because the dimension row may exist only now (late arriving dimension). Those rows are deleted from the fact table and recomputed from Silver in one transaction. The dimension keys come from an `ASOF JOIN` on the order time, so a customer or product change
affects only later orders.

In [ ]:
def gold_fact_sales(d):
    t0 = time.time()
    late = ""
    if has_table("gold", "fact_sales"):
        customers, products = late_dimension_keys(d)
        late = f"""
          UNION
          SELECT f.item_id FROM {fq('gold', 'fact_sales')} f JOIN {fq('silver', 'sales_order')} o ON o.order_id = f.order_id
          WHERE f.customer_key = -1 AND o.customer_id IN ({customers})
          UNION
          SELECT f.item_id FROM {fq('gold', 'fact_sales')} f JOIN {fq('silver', 'order_item')} i ON i.item_id = f.item_id
          WHERE f.product_key = -1 AND i.product_id IN ({products})"""
    q(f"""CREATE OR REPLACE TEMP TABLE _gi AS
          SELECT item_id FROM {fq('bronze', 'order_items')} WHERE _load_date = DATE '{d}'
          UNION
          SELECT item_id FROM {fq('silver', 'order_item')} WHERE order_id IN (SELECT order_id FROM {fq('bronze', 'orders')} WHERE _load_date = DATE '{d}'){late}""")
    q(f"""CREATE OR REPLACE TEMP TABLE _fs AS
          WITH it AS (
              SELECT i.item_id, i.order_id, i.product_id, i.quantity, i.unit_price, i.discount_pct, i.tax_rate, i.gross_amount, i.discount_amount, i.net_amount,
                     o.order_ts, o.customer_id, o.store_id, o.promotion_id, o.status AS order_status, o.channel
              FROM {fq('silver', 'order_item')} i JOIN {fq('silver', 'sales_order')} o ON o.order_id = i.order_id AND NOT o.is_deleted
              WHERE i.item_id IN (SELECT item_id FROM _gi))
          SELECT it.item_id, it.order_id, CAST(strftime(it.order_ts, '%Y%m%d') AS INTEGER) AS date_key,
                 COALESCE(dc.customer_key, -1) AS customer_key, COALESCE(dp.product_key, -1) AS product_key,
                 COALESCE(ds.store_key, -1) AS store_key, COALESCE(dm.promotion_key, -1) AS promotion_key,
                 it.order_status, it.channel, it.quantity, it.unit_price, it.discount_pct,
                 it.gross_amount, it.discount_amount, it.net_amount,
                 CAST(round(it.net_amount * it.tax_rate / 100.0, 2) AS DECIMAL(14, 2)) AS tax_amount
          FROM it
          ASOF LEFT JOIN {fq('gold', 'dim_customer')} dc ON dc.customer_id = it.customer_id AND it.order_ts >= dc.valid_from
          ASOF LEFT JOIN {fq('gold', 'dim_product')} dp ON dp.product_id = it.product_id AND it.order_ts >= dp.valid_from
          LEFT JOIN {fq('gold', 'dim_store')} ds ON ds.store_id = it.store_id
          LEFT JOIN {fq('gold', 'dim_promotion')} dm ON dm.promotion_id = it.promotion_id""")
    ensure_table("gold", "fact_sales", "SELECT * FROM _fs")
    target = fq("gold", "fact_sales")
    with tx():
        q(f"DELETE FROM {target} WHERE item_id IN (SELECT item_id FROM _gi)")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM _fs")
    log_step("gold", "fact_sales", d, rows_in=q("SELECT count(*) FROM _gi")[0][0], rows_out=q("SELECT count(*) FROM _fs")[0][0], t0=t0)

In [ ]:
gold_fact_sales(LOAD_DATE)

In [ ]:
def gold_fact_order(d):
    t0 = time.time()
    late = ""
    if has_table("gold", "fact_order"):
        customers, _ = late_dimension_keys(d)
        late = f"""
          UNION
          SELECT f.order_id FROM {fq('gold', 'fact_order')} f JOIN {fq('silver', 'sales_order')} o ON o.order_id = f.order_id
          WHERE f.customer_key = -1 AND o.customer_id IN ({customers})"""
    q(f"""CREATE OR REPLACE TEMP TABLE _go AS
          SELECT order_id FROM {fq('bronze', 'orders')} WHERE _load_date = DATE '{d}'
          UNION SELECT order_id FROM {fq('bronze', 'order_items')} WHERE _load_date = DATE '{d}'{late}""")
    q(f"""CREATE OR REPLACE TEMP TABLE _fo AS
          WITH o AS (SELECT * FROM {fq('silver', 'sales_order')} WHERE order_id IN (SELECT order_id FROM _go) AND NOT is_deleted),
               agg AS (SELECT order_id, count(*) AS item_count, sum(quantity) AS units, sum(gross_amount) AS gross_amount, sum(discount_amount) AS discount_amount,
                              sum(net_amount) AS net_amount
                       FROM {fq('silver', 'order_item')} WHERE order_id IN (SELECT order_id FROM _go) GROUP BY order_id)
          SELECT o.order_id, CAST(strftime(o.order_ts, '%Y%m%d') AS INTEGER) AS date_key,
                 COALESCE(dc.customer_key, -1) AS customer_key, COALESCE(ds.store_key, -1) AS store_key, COALESCE(dm.promotion_key, -1) AS promotion_key,
                 o.status AS order_status, o.channel, COALESCE(agg.item_count, 0) AS item_count, COALESCE(agg.units, 0) AS units,
                 COALESCE(agg.gross_amount, 0) AS gross_amount, COALESCE(agg.discount_amount, 0) AS discount_amount, COALESCE(agg.net_amount, 0) AS net_amount,
                 o.shipping_cost, o.order_total_reported,
                 CAST(o.order_total_reported - COALESCE(o.shipping_cost, 0) - COALESCE(agg.net_amount, 0) AS DECIMAL(14, 2)) AS total_difference,
                 COALESCE(agg.item_count, 0) = 0 AS has_no_items
          FROM o LEFT JOIN agg ON agg.order_id = o.order_id
          ASOF LEFT JOIN {fq('gold', 'dim_customer')} dc ON dc.customer_id = o.customer_id AND o.order_ts >= dc.valid_from
          LEFT JOIN {fq('gold', 'dim_store')} ds ON ds.store_id = o.store_id
          LEFT JOIN {fq('gold', 'dim_promotion')} dm ON dm.promotion_id = o.promotion_id""")
    ensure_table("gold", "fact_order", "SELECT * FROM _fo")
    target = fq("gold", "fact_order")
    with tx():
        q(f"DELETE FROM {target} WHERE order_id IN (SELECT order_id FROM _go)")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM _fo")
    log_step("gold", "fact_order", d, rows_in=q("SELECT count(*) FROM _go")[0][0], rows_out=q("SELECT count(*) FROM _fo")[0][0], t0=t0)

In [ ]:
gold_fact_order(LOAD_DATE)

In [ ]:
def gold_fact_payment(d):
    t0 = time.time()
    late = ""
    if has_table("gold", "fact_payment"):
        customers, _ = late_dimension_keys(d)
        late = f"""
          UNION
          SELECT f.payment_id FROM {fq('gold', 'fact_payment')} f JOIN {fq('silver', 'sales_order')} o ON o.order_id = f.order_id
          WHERE f.customer_key = -1 AND o.customer_id IN ({customers})"""
    q(f"""CREATE OR REPLACE TEMP TABLE _gp AS
          SELECT payment_id FROM {fq('bronze', 'payments')} WHERE _load_date = DATE '{d}'{late}""")
    q(f"""CREATE OR REPLACE TEMP TABLE _fp AS
          WITH p AS (SELECT p.*, o.order_ts, o.customer_id, o.total AS order_total
                     FROM {fq('silver', 'payment')} p
                     LEFT JOIN (SELECT order_id, order_ts, customer_id, order_total_reported AS total FROM {fq('silver', 'sales_order')}) o ON o.order_id = p.order_id
                     WHERE p.payment_id IN (SELECT payment_id FROM _gp))
          SELECT p.payment_id, p.order_id, CAST(strftime(p.paid_at, '%Y%m%d') AS INTEGER) AS date_key,
                 COALESCE(dc.customer_key, -1) AS customer_key, COALESCE(pm.payment_method_key, -1) AS payment_method_key,
                 p.status AS payment_status, p.amount, p.currency, CAST(p.amount - p.order_total AS DECIMAL(14, 2)) AS amount_vs_order_total
          FROM p
          ASOF LEFT JOIN {fq('gold', 'dim_customer')} dc ON dc.customer_id = p.customer_id AND p.paid_at >= dc.valid_from
          LEFT JOIN {fq('gold', 'dim_payment_method')} pm ON pm.payment_method = p.payment_method""")
    ensure_table("gold", "fact_payment", "SELECT * FROM _fp")
    target = fq("gold", "fact_payment")
    with tx():
        q(f"DELETE FROM {target} WHERE payment_id IN (SELECT payment_id FROM _gp)")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM _fp")
    log_step("gold", "fact_payment", d, rows_in=q("SELECT count(*) FROM _gp")[0][0], rows_out=q("SELECT count(*) FROM _fp")[0][0], t0=t0)

In [ ]:
gold_fact_payment(LOAD_DATE)

In [ ]:
def gold_fact_return(d):
    t0 = time.time()
    late = ""
    if has_table("gold", "fact_return"):
        customers, products = late_dimension_keys(d)
        late = f"""
          UNION
          SELECT f.return_id FROM {fq('gold', 'fact_return')} f JOIN {fq('silver', 'sales_order')} o ON o.order_id = f.order_id
          WHERE f.customer_key = -1 AND o.customer_id IN ({customers})
          UNION
          SELECT f.return_id FROM {fq('gold', 'fact_return')} f JOIN {fq('silver', 'order_item')} i ON i.item_id = f.item_id
          WHERE f.product_key = -1 AND i.product_id IN ({products})
          UNION
          SELECT f.return_id FROM {fq('gold', 'fact_return')} f
          WHERE f.item_id IN (SELECT item_id FROM {fq('bronze', 'order_items')} WHERE _load_date = DATE '{d}')"""
    q(f"""CREATE OR REPLACE TEMP TABLE _gr AS
          SELECT return_id FROM {fq('bronze', 'returns')} WHERE _load_date = DATE '{d}'{late}""")
    q(f"""CREATE OR REPLACE TEMP TABLE _fr AS
          WITH r AS (SELECT r.*, i.product_id, o.order_ts, o.customer_id
                     FROM {fq('silver', 'product_return')} r
                     LEFT JOIN {fq('silver', 'order_item')} i ON i.item_id = r.item_id
                     LEFT JOIN {fq('silver', 'sales_order')} o ON o.order_id = r.order_id
                     WHERE r.return_id IN (SELECT return_id FROM _gr))
          SELECT r.return_id, r.item_id, r.order_id, CAST(strftime(r.return_ts, '%Y%m%d') AS INTEGER) AS date_key,
                 COALESCE(dc.customer_key, -1) AS customer_key, COALESCE(dp.product_key, -1) AS product_key,
                 r.reason, r.status AS return_status, r.quantity, r.refund_amount
          FROM r
          ASOF LEFT JOIN {fq('gold', 'dim_customer')} dc ON dc.customer_id = r.customer_id AND COALESCE(r.order_ts, r.return_ts) >= dc.valid_from
          ASOF LEFT JOIN {fq('gold', 'dim_product')} dp ON dp.product_id = r.product_id AND COALESCE(r.order_ts, r.return_ts) >= dp.valid_from""")
    ensure_table("gold", "fact_return", "SELECT * FROM _fr")
    target = fq("gold", "fact_return")
    with tx():
        q(f"DELETE FROM {target} WHERE return_id IN (SELECT return_id FROM _gr)")
        q(f"INSERT INTO {target} BY NAME SELECT * FROM _fr")
    log_step("gold", "fact_return", d, rows_in=q("SELECT count(*) FROM _gr")[0][0], rows_out=q("SELECT count(*) FROM _fr")[0][0], t0=t0)

In [ ]:
gold_fact_return(LOAD_DATE)

### Gold views for reporting

Five views on top of the star schema, ready for any SQL/BI tool. They hold no data of their own and are created with `CREATE OR REPLACE VIEW` (a repeated run changes nothing in their content):

| View | Grain | What it answers |
|---|---|---|
| `v_sales_daily` | day | orders, items, units, gross, discount, net, tax and average order value per day |
| `v_sales_by_category_month` | month and product category | sales and **returns** (units, refunds, return rate) per category, with the category as it was when the item was sold (SCD2) |
| `v_store_performance_month` | month and store | orders, units and net sales per store, with type and region |
| `v_customer_value` | current customer | lifetime orders and net sales, first and last order, average order value, with segment and country |
| `v_payment_method_month` | month and payment method | payments, amount, failed and refunded payments, failure rate |

In [ ]:
GOLD_VIEWS = ["v_sales_daily", "v_sales_by_category_month", "v_store_performance_month", "v_customer_value", "v_payment_method_month"]


def gold_views(d):
    t0 = time.time()
    g = lambda t: fq("gold", t)
    views = {
        "v_sales_daily": f"""
            SELECT d.date, d.year, d.month, d.day_name, d.is_weekend,
                   count(DISTINCT f.order_id) AS orders, count(*) AS order_items, sum(f.quantity) AS units,
                   sum(f.gross_amount) AS gross_amount, sum(f.discount_amount) AS discount_amount, sum(f.net_amount) AS net_amount, sum(f.tax_amount) AS tax_amount,
                   round(sum(f.net_amount) / nullif(count(DISTINCT f.order_id), 0), 2) AS avg_order_value
            FROM {g('fact_sales')} f JOIN {g('dim_date')} d ON d.date_key = f.date_key
            GROUP BY d.date, d.year, d.month, d.day_name, d.is_weekend""",
        "v_sales_by_category_month": f"""
            WITH sales AS (
                SELECT d.year, d.month, p.category, count(DISTINCT f.order_id) AS orders, sum(f.quantity) AS units, sum(f.net_amount) AS net_amount
                FROM {g('fact_sales')} f JOIN {g('dim_product')} p ON p.product_key = f.product_key JOIN {g('dim_date')} d ON d.date_key = f.date_key
                GROUP BY d.year, d.month, p.category),
            returns AS (
                SELECT d.year, d.month, p.category, sum(r.quantity) AS returned_units, sum(r.refund_amount) AS refund_amount
                FROM {g('fact_return')} r JOIN {g('dim_product')} p ON p.product_key = r.product_key JOIN {g('dim_date')} d ON d.date_key = r.date_key
                GROUP BY d.year, d.month, p.category)
            SELECT s.year, s.month, s.category, s.orders, s.units, s.net_amount,
                   COALESCE(r.returned_units, 0) AS returned_units, COALESCE(r.refund_amount, 0) AS refund_amount,
                   round(COALESCE(r.returned_units, 0) * 100.0 / nullif(s.units, 0), 2) AS return_rate_pct
            FROM sales s LEFT JOIN returns r ON r.year = s.year AND r.month = s.month AND r.category = s.category""",
        "v_store_performance_month": f"""
            SELECT d.year, d.month, s.store_key, s.store_name, s.store_type, s.region, s.country,
                   count(DISTINCT f.order_id) AS orders, sum(f.quantity) AS units, sum(f.net_amount) AS net_amount
            FROM {g('fact_sales')} f JOIN {g('dim_store')} s ON s.store_key = f.store_key JOIN {g('dim_date')} d ON d.date_key = f.date_key
            GROUP BY d.year, d.month, s.store_key, s.store_name, s.store_type, s.region, s.country""",
        "v_customer_value": f"""
            WITH o AS (
                SELECT c.customer_id, count(*) AS orders, sum(fo.net_amount) AS net_amount, min(d.date) AS first_order, max(d.date) AS last_order
                FROM {g('fact_order')} fo JOIN {g('dim_customer')} c ON c.customer_key = fo.customer_key JOIN {g('dim_date')} d ON d.date_key = fo.date_key
                GROUP BY c.customer_id)
            SELECT c.customer_id, c.full_name, c.country, c.segment, c.birth_year,
                   COALESCE(o.orders, 0) AS orders, COALESCE(o.net_amount, 0) AS net_amount, o.first_order, o.last_order,
                   round(o.net_amount / nullif(o.orders, 0), 2) AS avg_order_value
            FROM {g('dim_customer')} c LEFT JOIN o ON o.customer_id = c.customer_id
            WHERE c.is_current AND c.customer_id > 0""",
        "v_payment_method_month": f"""
            SELECT d.year, d.month, m.payment_method, m.payment_method_name,
                   count(*) AS payments, sum(p.amount) AS amount,
                   count(*) FILTER (WHERE p.payment_status = 'failed') AS failed_payments,
                   round(count(*) FILTER (WHERE p.payment_status = 'failed') * 100.0 / count(*), 2) AS failure_rate_pct,
                   COALESCE(sum(p.amount) FILTER (WHERE p.payment_status = 'refunded'), 0) AS refunded_amount
            FROM {g('fact_payment')} p JOIN {g('dim_payment_method')} m ON m.payment_method_key = p.payment_method_key JOIN {g('dim_date')} d ON d.date_key = p.date_key
            GROUP BY d.year, d.month, m.payment_method, m.payment_method_name""",
    }
    assert list(views) == GOLD_VIEWS
    for name, select in views.items():
        q(f"CREATE OR REPLACE VIEW {g(name)} AS {select}")
    log_step("gold", "views", d, rows_out=len(views), t0=t0, note="(5 views)")

In [ ]:
gold_views(LOAD_DATE)

## Checks

Quality checks on the star schema. `expected` is what must hold; any row where `result` differs shows a problem. They look at the whole Gold layer as loaded so far.

In [ ]:
def scalar(sql):
    return q(sql)[0][0]


G = lambda t: fq("gold", t)
S = lambda t: fq("silver", t)

import pandas as pd


def quality_checks():
    """Run the quality checks on the Gold layer; returns one row per check."""
    checks = [
    ("fact_sales: duplicate item_id", scalar(f"SELECT count(*) - count(DISTINCT item_id) FROM {G('fact_sales')}"), 0),
    ("fact_order: duplicate order_id", scalar(f"SELECT count(*) - count(DISTINCT order_id) FROM {G('fact_order')}"), 0),
    ("fact_payment: duplicate payment_id", scalar(f"SELECT count(*) - count(DISTINCT payment_id) FROM {G('fact_payment')}"), 0),
    ("fact_return: duplicate return_id", scalar(f"SELECT count(*) - count(DISTINCT return_id) FROM {G('fact_return')}"), 0),
    ("fact_sales: customer_key without dimension row", scalar(f"SELECT count(*) FROM {G('fact_sales')} f LEFT JOIN {G('dim_customer')} d USING (customer_key) WHERE d.customer_key IS NULL"), 0),
    ("fact_sales: product_key without dimension row", scalar(f"SELECT count(*) FROM {G('fact_sales')} f LEFT JOIN {G('dim_product')} d USING (product_key) WHERE d.product_key IS NULL"), 0),
    ("fact_sales: store_key without dimension row", scalar(f"SELECT count(*) FROM {G('fact_sales')} f LEFT JOIN {G('dim_store')} d USING (store_key) WHERE d.store_key IS NULL"), 0),
    ("fact_sales: date_key without dimension row", scalar(f"SELECT count(*) FROM {G('fact_sales')} f LEFT JOIN {G('dim_date')} d USING (date_key) WHERE d.date_key IS NULL"), 0),
    ("dim_customer: more than one current row per customer", scalar(f"SELECT count(*) FROM (SELECT customer_id FROM {G('dim_customer')} WHERE is_current GROUP BY 1 HAVING count(*) > 1)"), 0),
    ("dim_customer: overlapping validity", scalar(f"SELECT count(*) FROM {G('dim_customer')} a JOIN {G('dim_customer')} b ON a.customer_id = b.customer_id AND a.customer_key < b.customer_key AND a.valid_from < b.valid_to AND b.valid_from < a.valid_to"), 0),
    ("dim_product: overlapping validity", scalar(f"SELECT count(*) FROM {G('dim_product')} a JOIN {G('dim_product')} b ON a.product_id = b.product_id AND a.product_key < b.product_key AND a.valid_from < b.valid_to AND b.valid_from < a.valid_to"), 0),
    ("fact_sales vs silver: net amount difference (non-deleted orders)",
     scalar(f"""SELECT CAST(coalesce((SELECT sum(net_amount) FROM {G('fact_sales')}), 0) - coalesce((SELECT sum(i.net_amount) FROM {S('order_item')} i JOIN {S('sales_order')} o USING (order_id) WHERE NOT o.is_deleted), 0) AS DECIMAL(18, 2))"""), 0),
    ("v_sales_daily: net amount differs from fact_sales",
     scalar(f"SELECT CAST(coalesce((SELECT sum(net_amount) FROM {G('v_sales_daily')}), 0) - coalesce((SELECT sum(net_amount) FROM {G('fact_sales')}), 0) AS DECIMAL(18, 2))"), 0),
    ]
    checks += [(f"view {v} returns rows", scalar(f"SELECT count(*) > 0 FROM {G(v)}"), True) for v in GOLD_VIEWS]
    result = pd.DataFrame(checks, columns=["check", "result", "expected"])
    result["ok"] = result["result"] == result["expected"]
    return result

### Overview of all layers

Row counts per table and the content of the quarantine.

In [ ]:
tables = q(f"SELECT table_schema, table_name, table_type FROM information_schema.tables WHERE table_catalog = '{CAT}' AND table_schema IN ('bronze', 'silver', 'gold') ORDER BY 1, 3 DESC, 2")
rows = [(s, t, kind.lower(), scalar(f"SELECT count(*) FROM {fq(s, t)}")) for s, t, kind in tables]
pd.DataFrame(rows, columns=["layer", "name", "type", "rows"])

In [ ]:
pd.DataFrame(q(f"SELECT source_table, reason, count(*) AS rows FROM {fq('silver', 'rejects')} GROUP BY ALL ORDER BY 1, 3 DESC"), columns=["source_table", "reason", "rows"])

## The whole pipeline as one function

`run_day(d)` runs all steps in the right order (Bronze for all tables, Silver, Gold dimensions, Gold facts, Gold views). The cells above did the same step by step; this function is used by the month loop
and the idempotency check below.

In [ ]:
def run_day(d):
    for t in BRONZE_TABLES:
        bronze_load(t, d)
    for name in ["customers", "addresses", "products", "suppliers", "stores", "promotions", "orders", "order_items", "payments", "returns"]:
        silver_cdc(name, d)
    gold_dim_payment_method(d)
    gold_dim_store(d)
    gold_dim_supplier(d)
    gold_dim_promotion(d)
    gold_dim_customer(d)
    gold_dim_product(d)
    gold_fact_sales(d)
    gold_fact_order(d)
    gold_fact_payment(d)
    gold_fact_return(d)
    gold_views(d)

## Load the whole month in a loop

To load **every day of the month, one day after the other**, set `LOAD_WHOLE_MONTH = True` in the first cell and run this section. The days are not typed in: the cell looks up which load dates exist in the Bronze folder
(`load_date=...` below `BRONZE`) and loads them in chronological order with `run_day`. As an alternative, `BACKFILL_FROM` and `BACKFILL_TO` (both inclusive) load a date range.

* Per day the loop prints one line: duration, rows read from Bronze and rows rejected in Silver. The detailed per-step output is only shown if a day fails.
* **It can be resumed.** `SKIP_LOADED_DAYS = True` skips days whose last step (the Gold views) is already in `ops.load_log`, so after an interruption the cell just continues with the first day that is missing. Set it to `False` to load every day again; because every
  step is repeatable, that gives the same result.
* A full month in the demo size takes roughly 15 to 25 minutes. When the loop is done, the quality checks run once more on the finished star schema.

With `LOAD_WHOLE_MONTH = True` you can skip the step-by-step cells for `LOAD_DATE` above (from "Bronze" to the Gold views) and run only the setup cells, the function definitions and this section.

In [ ]:
import contextlib
import io


def bronze_days():
    """All load dates that have files in the Bronze folder, oldest first."""
    rows = q(f"SELECT DISTINCT regexp_extract(file, 'load_date=([0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}})', 1) AS d FROM glob('{BRONZE}/*/*/*.parquet') ORDER BY 1")
    return [date.fromisoformat(r[0]) for r in rows if r[0]]


def loaded_days():
    """Days whose last pipeline step (the Gold views) is in the load log."""
    if not has_table("ops", "load_log"):
        return set()
    return {r[0] for r in q(f"SELECT load_date FROM {fq('ops', 'load_log')} WHERE layer = 'gold' AND table_name = 'views'")}


def load_days(days, skip_loaded=True):
    done = loaded_days() if skip_loaded else set()
    todo = [d for d in days if d not in done]
    print(f"{len(days)} days selected, {len(days) - len(todo)} already loaded and skipped, {len(todo)} to load")
    started = time.time()
    for i, d in enumerate(todo, 1):
        t0 = time.time()
        output = io.StringIO()
        try:
            with contextlib.redirect_stdout(output):
                run_day(d)
        except BaseException:
            print(output.getvalue())
            print(f"\n!!! {d} failed - fix the cause and run this cell again: the days before it are skipped")
            raise
        secs = time.time() - t0
        rows, rejected = q(f"SELECT coalesce(sum(CASE WHEN layer = 'bronze' THEN rows_out END), 0), coalesce(sum(CASE WHEN layer = 'silver' THEN rows_rejected END), 0) "
                           f"FROM {fq('ops', 'load_log')} WHERE load_date = DATE '{d}'")[0]
        left = (time.time() - started) / i * (len(todo) - i)
        print(f"[{i:>2}/{len(todo)}] {d}  {secs:5.1f}s  bronze rows {int(rows):>10,}  silver rejects {int(rejected):>7,}  about {left / 60:4.1f} min left")
    print(f"\nDone: {len(todo)} days in {(time.time() - started) / 60:.1f} min")


if LOAD_WHOLE_MONTH:
    selected = bronze_days()
elif BACKFILL_FROM and BACKFILL_TO:
    selected = [BACKFILL_FROM + timedelta(days=n) for n in range((BACKFILL_TO - BACKFILL_FROM).days + 1)]
else:
    selected = []

if selected:
    load_days(selected, SKIP_LOADED_DAYS)
    display(quality_checks())
else:
    print("Nothing selected: set LOAD_WHOLE_MONTH = True (or BACKFILL_FROM and BACKFILL_TO) in the first cell to load several days in a loop.")

## Idempotency proof

`fingerprint` computes the row count and an order-independent hash of the full content of every Bronze, Silver and Gold table and view. The next cell takes a fingerprint, **runs the complete pipeline for `LOAD_DATE`
again**, takes a second fingerprint and compares. They must be identical: no duplicates, no changed values. (The DuckLake snapshots do change, because every run rewrites the load date's rows.) The proof is most convincing
after the month loop above: `LOAD_DATE` can then be any day in the middle of the month, with all later days already loaded. Even then nothing changes, because every business key is recomputed from the whole Bronze history (later updates and deletes stay applied) and facts that point to the unknown member are repaired when the missing dimension row arrives.

In [ ]:
def fingerprint():
    out = {}
    for s in ("silver", "gold"):
        for _, t in q(f"SELECT table_schema, table_name FROM information_schema.tables WHERE table_catalog = '{CAT}' AND table_schema = '{s}' ORDER BY 2"):
            if t == "rejects":
                sql = f"SELECT count(*), sum(CAST(hash(source_table, record_key, reason, updated_at, load_date) AS HUGEINT)) FROM {fq(s, t)}"
            else:
                sql = f"SELECT count(*), sum(CAST(hash(x) AS HUGEINT)) FROM {fq(s, t)} x"
            out[f"{s}.{t}"] = q(sql)[0]
    for t in BRONZE_TABLES:
        if has_table("bronze", t):
            out[f"bronze.{t}"] = q(f"SELECT count(*), sum(CAST(hash(x) AS HUGEINT)) FROM {fq('bronze', t)} x")[0]
    return out

In [ ]:
before = fingerprint()
run_day(LOAD_DATE)
after = fingerprint()
diff = {k: (before.get(k), after.get(k)) for k in after if before.get(k) != after.get(k)}
print("\nIdempotency check for", LOAD_DATE, "->", "IDENTICAL (nothing changed)" if not diff else f"CHANGED: {diff}")

## Load log

In [ ]:
pd.DataFrame(q(f"SELECT layer, table_name, load_date, rows_in, rows_out, rows_rejected, rows_deleted, seconds FROM {fq('ops', 'load_log')} ORDER BY load_date DESC, layer, table_name LIMIT 60"),
             columns=["layer", "table", "load_date", "rows_in", "rows_out", "rejected", "deleted", "seconds"])

## DuckLake snapshots

Every transaction of the pipeline is a DuckLake snapshot; `changes` shows what it did (tables created, inserted into, deleted from, columns added).

In [ ]:
pd.DataFrame(q(f"SELECT snapshot_id, snapshot_time, schema_version, changes FROM ducklake_snapshots('{CAT}') ORDER BY snapshot_id DESC LIMIT 12"),
             columns=["snapshot_id", "snapshot_time", "schema_version", "changes"])